In [ ]:
# 配置格：只读后台续跑共用的参数文件，不执行模型。
import json
from project import resolve_root
from preparation.images.annotation.image_annotation_pipeline import config, run_pipeline

ROOT = resolve_root()
RUN = 'neutral_images_v2_20261001'
CONFIG_PATH = ROOT / 'demiwtg/preparation/images/annotation/runs' / RUN / 'config.json'
# 参数在此 JSON 中维护：原 ready 名单、images@16、DP2、through=export。
# 正式运行 valid_rows：自动将成功描述与丰富度合并到公共表，失败行保留。
CONFIG = config(**json.loads(CONFIG_PATH.read_text()))
RUN_PIPELINE = False
print('参数文件：', CONFIG_PATH)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))


In [ ]:
# 执行格：手工开启后调用唯一入口；不会启动概念评分或其他 pipeline。
if RUN_PIPELINE:
    RESULT = run_pipeline(CONFIG)
    print('complete:', RESULT['complete'], 'committed:', RESULT['committed'])
else:
    print('未执行；当前配置仅显示。')


In [ ]:
# 只读格：可在新 kernel 独立运行，只展示当前 summary 指向的固定结果。
from IPython.display import HTML, display
from project import resolve_root
from preparation.images.annotation.operaters import preview

ROOT = resolve_root()
VIEW_RUN = 'neutral_images_v2_20261001'
display(preview.stage_panels(ROOT, VIEW_RUN))
image_frame = preview.image_result_frame(ROOT, VIEW_RUN)
print('当前摘要引用的图片结果：', len(image_frame))
if len(image_frame):
    display(image_frame['status'].value_counts())
    display(image_frame.head(50))
    PAGE, PAGE_SIZE = 1, 20
    display(HTML(preview.thumbnail_page(ROOT, image_frame.to_dict('records'), page=PAGE, page_size=PAGE_SIZE)))
else:
    print('尚无本轮已提交的图片结果；不以旧阶段或进程状态冒充完成。')


In [ ]:
# 只读状态格：独立于配置/执行格，区别完成、公共提交和失败数量。
from IPython.display import display
from project import resolve_root
from preparation.images.annotation.operaters import preview

VIEW_RUN = 'neutral_images_v2_20261001'
summary, frame, state, artifacts = preview.run_progress(resolve_root(), VIEW_RUN)
display(frame)
print('本轮状态与固定结果：', state)
print('运行目录产物：', artifacts)
if not summary:
    print('尚未提交 summary，不能判定成功。')
